# Week 8 – Business Question

**Nama:** Noviana Putri Yuliani  
**Sekolah:** SMK Negeri 2 Surakarta  
**Jurusan:** TJKT

## 2. Business Problem

**Business Problem:**  
PT Edusoft Center Teknologi menghadapi penurunan dan perubahan kinerja penjualan pada periode pengamatan.

**Catatan penting:**  
Business case menyebut penjualan mengalami penurunan selama 3 bulan terakhir. Namun, hasil aktual dari dataset menunjukkan penjualan tidak turun secara terus-menerus. Sales Januari sebesar 116.291,87 turun menjadi 97.219,37 pada Februari atau turun 16,40%, kemudian naik menjadi 109.455,51 pada Maret atau naik 12,59% dibanding Februari. Oleh karena itu, narasi penurunan digunakan sebagai business case, sedangkan kondisi aktual mengikuti hasil analisis dataset.

Dataset **Supermarket Sales Cleaned.xlsx** hanya digunakan sebagai bahan analisis/evidence, bukan sebagai data operasional asli PT Edusoft.

## 3. Business Questions

1. Bagaimana tren sales dari Januari sampai Maret 2019?
2. Bagaimana perubahan jumlah transaksi dan AOV berkaitan dengan perubahan sales?
3. Bagaimana perbedaan kontribusi penjualan antarproduct line?
4. Bagaimana perbedaan performa penjualan antarbranch?
5. Bagaimana karakteristik transaksi berdasarkan customer type, gender, payment, dan quantity?

## 4. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

## 5. Load Dataset

File yang digunakan adalah **Supermarket Sales Cleaned.xlsx**. Dataset ini digunakan sebagai dataset analisis/evidence untuk business case, bukan sebagai data operasional asli PT Edusoft.

In [ ]:
file_path = "Supermarket Sales Cleaned.xlsx"
df = pd.read_excel(file_path)

df.head()

## 6. Dataset Overview

Menurut dokumen Week 8, dataset yang digunakan terdiri dari **1.000 transaksi** dan **21 kolom**, dengan periode Januari–Maret 2019. Jumlah baris dan kolom juga dicek langsung dari dataset agar hasil yang tampil mengikuti file yang digunakan.

In [ ]:
print("Shape dataset:", df.shape)
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("\nNama kolom:")
print(df.columns.tolist())

print("\n5 baris awal:")
display(df.head())

## 7. Data Quality Check

In [ ]:
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate Invoice ID:", int(df["Invoice ID"].duplicated().sum()))
print("Unique Invoice ID:", int(df["Invoice ID"].nunique()))

## 8. Missing Value Check

In [ ]:
missing = df.isna().sum()
print("Total missing value:", int(missing.sum()))
display(missing[missing > 0].to_frame("Jumlah Missing"))

## 9. Data Type Check

In [ ]:
display(df.dtypes.to_frame("Data Type"))

## 10. Descriptive Statistics

Statistik deskriptif digunakan untuk melihat gambaran umum kolom numerik yang relevan. Hasilnya tidak digunakan untuk menyimpulkan sebab-akibat.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
desc = df[numeric_cols].describe().T
display(desc.round(2))

## 11. KPI Calculation

KPI yang digunakan:
- **Total Sales** = SUM(Total)
- **Number of Transactions** = COUNT DISTINCT Invoice ID
- **Quantity Sold** = SUM(Quantity)
- **Gross Income** = SUM(gross income)
- **AOV** = Total Sales / Number of Transactions

Semua angka KPI ditampilkan dengan 2 angka desimal.

In [ ]:
total_sales = df["Total"].sum()
transactions = df["Invoice ID"].nunique()
quantity_sold = df["Quantity"].sum()
gross_income = df["gross income"].sum()
aov = total_sales / transactions

kpi = pd.DataFrame({
    "KPI": ["Total Sales", "Number of Transactions", "Quantity Sold", "Gross Income", "AOV"],
    "Value": [total_sales, transactions, quantity_sold, gross_income, aov]
})

display(kpi.style.format({"Value": "{:,.2f}"}))

## 12. Monthly Sales Analysis

Analisis bulanan menggunakan **Year** dan **Month**, lalu menampilkan Sales, Transactions, Quantity, AOV, dan Sales Growth.

In [ ]:
monthly = (
    df.groupby(["Year", "Month"])
      .agg(
          Sales=("Total", "sum"),
          Transactions=("Invoice ID", "nunique"),
          Quantity=("Quantity", "sum")
      )
      .reset_index()
)

monthly["AOV"] = monthly["Sales"] / monthly["Transactions"]
monthly["Sales Growth"] = monthly["Sales"].pct_change() * 100

display(monthly.round(2))

## 13. Sales Growth / Decline

Formula:

**(Current Sales - Previous Sales) / Previous Sales × 100%**

Hasil:
- Januari → Februari: sales turun **16,40%**
- Februari → Maret: sales naik **12,59%**

Jadi, sales tidak turun selama tiga bulan berturut-turut. Penurunan paling jelas terjadi pada Februari dibanding Januari, sedangkan Maret menunjukkan pemulihan dibanding Februari tetapi belum kembali ke sales Januari.

In [ ]:
growth_table = monthly[["Year", "Month", "Sales", "Sales Growth"]].copy()
display(growth_table.round(2))

## 14. Transaction & AOV Analysis

Untuk Januari → Februari:
- Transactions: **352 → 303**
- AOV: **330,37 → 320,86**
- Sales: **116.291,87 → 97.219,37**

Penurunan sales pada Februari terjadi bersamaan dengan penurunan jumlah transaksi dan AOV. Data ini menunjukkan bahwa kedua metrik tersebut menjadi driver/evidence dari perubahan sales, tetapi tidak membuktikan penyebab operasional di baliknya.

In [ ]:
transaction_aov = monthly[["Year", "Month", "Sales", "Transactions", "AOV"]].copy()
display(transaction_aov.round(2))

## 15. Product Line Analysis

Analisis product line mencakup Sales, Transactions, Quantity, AOV, Contribution %, dan Cumulative %.

**Finding:** Health and beauty memiliki sales terendah sehingga menjadi area yang perlu diperhatikan lebih lanjut. Hal ini bukan berarti Health and beauty menjadi penyebab penurunan sales.

In [ ]:
product = (
    df.groupby("Product line")
      .agg(
          Sales=("Total", "sum"),
          Transactions=("Invoice ID", "nunique"),
          Quantity=("Quantity", "sum")
      )
      .sort_values("Sales", ascending=False)
      .reset_index()
)

product["AOV"] = product["Sales"] / product["Transactions"]
product["Contribution %"] = product["Sales"] / product["Sales"].sum() * 100
product["Cumulative %"] = product["Contribution %"].cumsum()

display(product.round(2))

## 16. Pareto / Sales Contribution

Pareto digunakan untuk melihat konsentrasi kontribusi sales, bukan untuk menetapkan penyebab penurunan.

Tiga product line dengan kontribusi terbesar adalah:
1. Food and beverages
2. Sports and travel
3. Electronic accessories

Ketiganya menyumbang sekitar **51,28%** dari total sales.

In [ ]:
top3_contribution = product.head(3)["Contribution %"].sum()
print(f"Kontribusi Top 3 Product Line: {top3_contribution:.2f}%")
display(product[["Product line", "Sales", "Contribution %", "Cumulative %"]].round(2))

## 17. Branch Analysis

Analisis branch dikelompokkan berdasarkan **Branch** dan **City**.

**Finding:** Branch C / Naypyitaw memiliki sales, contribution, dan AOV tertinggi. Branch C adalah evidence/finding, bukan root cause.

In [ ]:
branch = (
    df.groupby(["Branch", "City"])
      .agg(
          Sales=("Total", "sum"),
          Transactions=("Invoice ID", "nunique"),
          Quantity=("Quantity", "sum")
      )
      .reset_index()
)

branch["AOV"] = branch["Sales"] / branch["Transactions"]
branch["Contribution %"] = branch["Sales"] / branch["Sales"].sum() * 100

display(branch.round(2))

## 18. Customer / Gender / Payment / Quantity Analysis

Analisis ini digunakan untuk melihat karakteristik transaksi dan perbedaan antarsegmen berdasarkan customer type, gender, payment, dan quantity.

Dataset tidak memiliki **Customer ID**, sehingga analisis tidak digunakan untuk menyimpulkan customer retention atau customer loss.

In [ ]:
for col in ["Customer type", "Gender", "Payment"]:
    summary = (
        df.groupby(col)
          .agg(
              Sales=("Total", "sum"),
              Transactions=("Invoice ID", "nunique"),
              Quantity=("Quantity", "sum")
          )
          .reset_index()
    )
    print(f"\n### {col}")
    display(summary.round(2))

print("\n### Quantity")
quantity_summary = (
    df.groupby("Quantity")
      .agg(
          Sales=("Total", "sum"),
          Transactions=("Invoice ID", "nunique")
      )
      .reset_index()
)
display(quantity_summary.round(2))

## 19. Visualizations

Visualisasi yang digunakan sesuai Business Questions:
1. Line chart monthly sales
2. Bar chart sales by product line
3. Bar chart sales by branch

In [ ]:
plt.figure(figsize=(8, 5))
labels = monthly["Month"].astype(str)
plt.plot(labels, monthly["Sales"], marker="o")
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(product["Product line"], product["Sales"])
plt.title("Sales by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Sales")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
branch_labels = branch["Branch"] + " / " + branch["City"]
plt.bar(branch_labels, branch["Sales"])
plt.title("Sales by Branch")
plt.xlabel("Branch / City")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

## 20. Problem Identification

Tiga problem yang digunakan adalah:

**Problem 1:** Sales Februari lebih rendah dibanding Januari.  
**Problem 2:** Performa penjualan antarproduct line berbeda.  
**Problem 3:** Performa penjualan antarbranch berbeda.

Transactions dan AOV digunakan sebagai evidence/driver untuk Problem 1, bukan sebagai problem terpisah maupun root cause.

## 21. Evidence Data

Evidence utama:
1. Sales Januari = **116.291,87** dan Sales Februari = **97.219,37**.
2. Sales Januari → Februari turun **16,40%**.
3. Transactions Januari → Februari: **352 → 303**.
4. AOV Januari → Februari: **330,37 → 320,86**.
5. Top 3 product line menyumbang **51,28%** sales.
6. Branch C memiliki contribution tertinggi sebesar **34,24%**.

Evidence bukan root cause.

In [ ]:
evidence = pd.DataFrame({
    "Evidence": [
        "Sales Januari",
        "Sales Februari",
        "Sales Growth Jan → Feb",
        "Transactions Jan → Feb",
        "AOV Jan → Feb",
        "Top 3 Product Line Contribution",
        "Branch C Contribution"
    ],
    "Value": [
        monthly.loc[monthly["Month"] == 1, "Sales"].iloc[0],
        monthly.loc[monthly["Month"] == 2, "Sales"].iloc[0],
        monthly.loc[monthly["Month"] == 2, "Sales Growth"].iloc[0],
        "352 → 303",
        "330,37 → 320,86",
        top3_contribution,
        branch.loc[branch["Branch"] == "C", "Contribution %"].iloc[0]
    ]
})
display(evidence)

## 22. Potential Cause Analysis

Potential cause berikut semuanya berstatus **Potential Cause – membutuhkan validasi data tambahan**:

1. Promotion
2. Price
3. Stock Availability
4. Customer Behavior / Retention
5. Service / Process
6. Competition / External Factors

Data yang belum tersedia:
- Promotion Flag / data promosi
- Stock level / stockout
- Customer ID dan histori pelanggan
- Data pelayanan / complaint / antrean
- Data kompetitor / external factors

Untuk **price**, Unit Price tersedia, tetapi riwayat perubahan harga dan strategi harga tidak tersedia secara lengkap sehingga belum cukup untuk menyimpulkan price sebagai penyebab.

In [ ]:
potential_causes = pd.DataFrame({
    "Potential Cause": [
        "Promotion",
        "Price",
        "Stock Availability",
        "Customer Behavior / Retention",
        "Service / Process",
        "Competition / External Factors"
    ],
    "Status": ["Potential Cause – membutuhkan validasi data tambahan."] * 6
})
display(potential_causes)

## 23. Root Cause Analysis / Five Whys

**Why 1:** Mengapa performa sales berubah?  
Karena sales Februari lebih rendah dibanding Januari.

**Why 2:** Mengapa sales Februari lebih rendah?  
Karena jumlah transaksi turun dan AOV juga turun.

**Why 3:** Mengapa transaksi dan AOV turun?  
Dataset menunjukkan perubahan tersebut, tetapi tidak menjelaskan penyebab operasionalnya.

**Why 4:** Mengapa penyebab operasional belum dapat dipastikan?  
Karena tidak tersedia data promosi, stok, Customer ID, kompetitor, pelayanan, atau alasan transaksi batal.

**Why 5:** Mengapa data tersebut diperlukan?  
Karena validasi root cause membutuhkan bukti tambahan di luar angka sales, transaksi, dan AOV.

**Kesimpulan:**  
Root cause sebenarnya belum dapat ditetapkan hanya dari dataset ini. Faktor promosi, harga, stok, customer behavior, service/process, dan competition masih berstatus potential cause dan membutuhkan data tambahan untuk validasi.

## 24. Insights

1. Penurunan sales paling jelas terjadi pada Februari dibanding Januari, kemudian sales kembali naik pada Maret.
2. Penurunan Februari terjadi bersamaan dengan penurunan jumlah transaksi dan AOV.
3. Kontribusi sales antarproduct line berbeda, dengan tiga product line terbesar menyumbang sekitar 51,28% total sales.
4. Branch C memiliki sales dan AOV tertinggi dengan kontribusi 34,24%.

Insight tersebut bukan root cause.

## 25. Business Implication

Perubahan sales dapat berdampak pada:
- performa penjualan,
- pendapatan,
- evaluasi produk,
- evaluasi cabang,
- pengambilan keputusan operasional.

Interpretasi tetap bersifat deskriptif dan tidak menyatakan hubungan sebab-akibat yang belum terbukti.

## 26. Recommendation

1. Evaluasi penyebab turunnya transaksi pada Februari.
2. Evaluasi AOV dan pola pembelian pelanggan.
3. Perhatikan product line dengan kontribusi rendah.
4. Pertahankan performa product line dengan kontribusi tinggi.
5. Evaluasi performa branch secara berkala.
6. Tambahkan data promosi, stok, Customer ID, service/complaint, dan kompetitor untuk analisis lanjutan.
7. Lakukan monitoring sales secara berkala agar penurunan dapat diketahui lebih awal.

Rekomendasi di atas tidak menganggap potential cause sebagai penyebab yang sudah terbukti.

## 27. Limitation

- Dataset bukan data operasional asli PT Edusoft.
- Dataset hanya digunakan sebagai bahan analisis/evidence untuk business case.
- Tidak tersedia Promotion Flag.
- Tidak tersedia Stock Level/Stockout.
- Tidak tersedia Customer ID.
- Tidak tersedia histori customer.
- Tidak tersedia data service/complaint.
- Tidak tersedia data kompetitor/external factors.
- Oleh karena itu root cause tidak dapat dipastikan hanya dari dataset.

## 28. Conclusion

Analisis menunjukkan bahwa sales mengalami penurunan pada Februari sebesar 16,40% dibanding Januari, kemudian meningkat 12,59% pada Maret dibanding Februari. Penurunan Februari terjadi bersamaan dengan penurunan jumlah transaksi dan AOV. Performa product line dan branch juga menunjukkan perbedaan kontribusi. Namun, dataset belum menyediakan data yang cukup untuk memastikan root cause seperti promosi, stok, perubahan harga, customer retention, pelayanan, atau kompetisi. Oleh karena itu, faktor-faktor tersebut masih menjadi potential cause yang membutuhkan validasi dengan data tambahan.

### Konsistensi Week 8

Alur analisis yang digunakan:

**Business Problem → Business Question → KPI → Finding / Evidence → Problem → Potential Cause → Root Cause Validation → Insight → Recommendation**

Dataset Supermarket Sales digunakan sebagai evidence untuk business case PT Edusoft Center Teknologi, bukan sebagai data operasional asli PT Edusoft.